# GNN Bipartite Tertile — v2 (weighted SAGE + GATv2 attention on w·log_aum)

Changes vs `gnn_bipartite_tertile_parquet.ipynb`:

1. **SAGE** now passes `edge_weight` into every `SAGEConv` call, so the signed Δ-weight actually scales messages.
2. **GAT** is replaced by **GATv2** with `edge_dim=1`; the per-edge feature is `edge_weight × log_aum(source CIK)`, so attention conditions on "how big the change is" × "how big the fund is".
3. After each quarterly two-periods evaluation, ranks the test-graph CUSIPs by `P(top tertile)` and appends `(cusip, year, quarter, score)` to a parquet file.
4. The whole sweep is wrapped in a loop over both edge columns (`change_in_weight`, `change_in_adjusted_weight`); per-run artifacts are suffixed by the column name so the two runs don't overwrite each other.

## Setup

### torch / cuda check

In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    print("cuda build:", torch.version.cuda)


### Imports

In [ ]:
import os
import inspect
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv, GATv2Conv, GraphConv

# PyG < 2.4's SAGEConv has no `edge_weight` arg (3rd positional is `size`),
# which is the root cause of the "expected size <float>" error. When SAGEConv
# can't take edge_weight, fall back to GraphConv -- same mean-aggregation
# message-passing structure, but with native edge_weight support.
SAGE_SUPPORTS_EW = "edge_weight" in inspect.signature(SAGEConv.forward).parameters

warnings.filterwarnings("ignore", category=UserWarning)
print(f"[v2] SAGEConv supports edge_weight: {SAGE_SUPPORTS_EW}"
      + ("" if SAGE_SUPPORTS_EW else "  -> falling back to GraphConv for WeightedSAGE"))


In [ ]:
OUT_DIR = Path(os.environ.get("FGNN_OUT_DIR", str(Path.home() / "13Fgnn"))).expanduser().resolve()
DATA_DIR = Path(os.environ.get("FGNN_DATA_DIR", str(OUT_DIR / "data"))).expanduser().resolve()
MODELS_DIR = OUT_DIR / "models"
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"DATA_DIR not found: {DATA_DIR}. Set FGNN_DATA_DIR.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.05)
torch.manual_seed(0)
np.random.seed(0)

print("OUT_DIR    :", OUT_DIR)
print("DATA_DIR   :", DATA_DIR)
print("MODELS_DIR :", MODELS_DIR)
print("device     :", DEVICE)


### Load parquets

In [ ]:
TGT_CHANGED_HOLDINGS = "changed_holdings"
TGT_STOCKS_RETURN = "stocks_return"
TGT_CIK_AUM = "cik_aum"
TGT_NORMALIZED_HOLDINGS = "normalized_holdings"
CUSIP_FIN_TABLE = "cusip_financial_data"


def _read_parquet_or_empty(name):
    p = DATA_DIR / f"{name}.parquet"
    if not p.exists():
        print(f"  [warn] missing parquet: {p.name} (returning empty df)")
        return pd.DataFrame()
    df = pd.read_parquet(p)
    print(f"  loaded {name:30s} {len(df):>10,} rows  {len(df.columns):>3} cols")
    return df


CHANGED_HOLDINGS = _read_parquet_or_empty(TGT_CHANGED_HOLDINGS)
STOCKS_RETURN = _read_parquet_or_empty(TGT_STOCKS_RETURN)
CIK_AUM = _read_parquet_or_empty(TGT_CIK_AUM)
NORM_HOLDINGS = _read_parquet_or_empty(TGT_NORMALIZED_HOLDINGS)
CUSIP_FIN = _read_parquet_or_empty(CUSIP_FIN_TABLE)


### Configuration

In [ ]:
STOCK_FEATURE_COLS = [
    "diluted_eps", "roe", "ev_ebitda", "pe_ratio", "price_to_sales",
    "price_to_book", "debt_to_equity", "dividend_yield", "fcf_per_share",
    "log_return",
]

# Run the full sweep once for each of these edge-weight columns.
EDGES_COLUMN_NAMES = ["change_in_weight", "change_in_adjusted_weight"]

YEAR, QUARTER = 2019, 3

NUM_CLASSES = 3
HIDDEN_DIM = 64
NUM_LAYERS = 2
EPOCHS = 150
LR = 8e-4
DROPOUT = 0.5
MASK_FRAC = 0.20
GAT_HEADS = 4


## Data loaders

In [ ]:
def next_year_quarter(year, quarter):
    return (year + 1, 1) if quarter == 4 else (year, quarter + 1)


def prev_year_quarter(year, quarter):
    return (year - 1, 4) if quarter == 1 else (year, quarter - 1)


def load_edges(year, quarter, edges_col_name):
    df = CHANGED_HOLDINGS
    if df.empty or edges_col_name not in df.columns:
        return pd.DataFrame(columns=["cik", "cusip", "w"])
    mask = (df["year"] == year) & (df["quarter"] == quarter) & df[edges_col_name].notna()
    return (df.loc[mask, ["cik", "cusip", edges_col_name]]
             .rename(columns={edges_col_name: "w"})
             .reset_index(drop=True))


def load_returns(year, quarter):
    df = STOCKS_RETURN
    if df.empty:
        return pd.DataFrame(columns=["cusip", "log_return"])
    mask = (df["year"] == year) & (df["quarter"] == quarter) & df["log_return"].notna()
    return df.loc[mask, ["cusip", "log_return"]].reset_index(drop=True)


def load_aum(year, quarter):
    df = CIK_AUM
    if df.empty:
        return pd.DataFrame(columns=["cik", "aum"])
    mask = (df["year"] == year) & (df["quarter"] == quarter) & (df["total"] > 0)
    return (df.loc[mask, ["cik", "total"]]
             .rename(columns={"total": "aum"})
             .reset_index(drop=True))


def load_stock_features(year, quarter):
    fin = CUSIP_FIN
    if fin.empty:
        fin = pd.DataFrame(columns=["cusip"] + STOCK_FEATURE_COLS)
    else:
        fin = fin.loc[(fin["year"] == year) & (fin["quarter"] == quarter)].copy()
    rets = load_returns(year, quarter)
    df = fin.merge(rets, on="cusip", how="outer")
    for c in STOCK_FEATURE_COLS:
        if c not in df.columns:
            df[c] = 0.0
    return df[["cusip"] + STOCK_FEATURE_COLS]


def investor_profitability(year, quarter):
    ny, nq = next_year_quarter(year, quarter)
    h = NORM_HOLDINGS
    if h.empty:
        return pd.Series(dtype=float, name="profitability")
    h = h.loc[(h["year"] == year) & (h["quarter"] == quarter), ["cik", "cusip", "weight"]]
    r = load_returns(ny, nq)
    m = h.merge(r, on="cusip", how="inner")
    m["contrib"] = m["weight"] * m["log_return"]
    return m.groupby("cik")["contrib"].sum().rename("profitability")


## Tertile labeller & z-score

In [ ]:
def tertile_labels(values):
    s = values.astype(float)
    out = pd.Series(-1, index=s.index, dtype=np.int64)
    valid = s.dropna()
    if valid.empty:
        return out
    try:
        cats = pd.qcut(valid, q=3, labels=[0, 1, 2])
    except ValueError:
        q1, q2 = np.quantile(valid, [1 / 3, 2 / 3])
        cats = pd.cut(valid, bins=[-np.inf, q1, q2, np.inf], labels=[0, 1, 2], include_lowest=True)
    out.loc[valid.index] = cats.astype(np.int64)
    return out


def zscore(df, cols):
    out = df.copy()
    for c in cols:
        v = out[c].astype(float)
        v = v.replace([np.inf, -np.inf], np.nan).fillna(v.median() if v.notna().any() else 0.0)
        sd = v.std()
        out[c] = (v - v.mean()) / sd if sd > 0 else 0.0
    return out


## Graph builder (with `edge_attr = w · log_aum`)

In [ ]:
def _build_cik_features(edges, year, quarter):
    """One row per CIK with [log_aum, n_holdings, profitability], z-scored."""
    aum = load_aum(year, quarter)
    py, pq = prev_year_quarter(year, quarter)
    try:
        past_prof = investor_profitability(py, pq).reset_index()
    except Exception:
        past_prof = pd.DataFrame(columns=["cik", "profitability"])

    cik_nh = edges.groupby("cik").size().rename("n_holdings").reset_index()
    cik_df = (aum.merge(cik_nh, on="cik", how="outer")
                  .merge(past_prof, on="cik", how="left"))
    cik_df["aum"] = cik_df["aum"].fillna(
        cik_df["aum"].median() if cik_df["aum"].notna().any() else 0.0)
    cik_df["log_aum"] = np.log(cik_df["aum"].clip(lower=1.0))
    cik_df["n_holdings"] = cik_df["n_holdings"].fillna(0)
    cik_df["profitability"] = cik_df["profitability"].fillna(0.0)

    CIK_FEATS = ["log_aum", "n_holdings", "profitability"]
    return zscore(cik_df, CIK_FEATS), CIK_FEATS


def _build_stock_features(year, quarter):
    return zscore(load_stock_features(year, quarter), STOCK_FEATURE_COLS)


def _load_label_sources(year, quarter):
    ny, nq = next_year_quarter(year, quarter)
    r_next = load_returns(ny, nq).set_index("cusip")["log_return"]
    prof_next = investor_profitability(year, quarter)
    return r_next, prof_next


def _assemble_node_matrix(edges, cik_df, stock_df, CIK_FEATS):
    cik_ids = pd.Index(edges["cik"].unique())
    cusip_ids = pd.Index(edges["cusip"].unique())
    cik_df = cik_df.set_index("cik").reindex(cik_ids).fillna(0.0)
    stock_df = stock_df.set_index("cusip").reindex(cusip_ids).fillna(0.0)
    F_cik = cik_df[CIK_FEATS].to_numpy()
    F_stk = stock_df[STOCK_FEATURE_COLS].to_numpy()
    Fdim = F_cik.shape[1] + F_stk.shape[1]
    X = np.zeros((len(cik_ids) + len(cusip_ids), Fdim), dtype=np.float32)
    X[:len(cik_ids), :F_cik.shape[1]] = F_cik
    X[len(cik_ids):, F_cik.shape[1]:] = F_stk
    cik_pos = {c: i for i, c in enumerate(cik_ids)}
    cusip_pos = {c: i + len(cik_ids) for i, c in enumerate(cusip_ids)}
    return X, cik_ids, cusip_ids, cik_pos, cusip_pos


def _assemble_edges(edges, cik_pos, cusip_pos, cik_df):
    """Returns edge_index (2, 2E), edge_weight (2E,), edge_attr (2E, 1).

    edge_attr = (signed Δ-weight) * (z-scored log_aum of the source CIK).
    GATv2 uses this scalar in its attention computation.
    """
    src = edges["cik"].map(cik_pos).to_numpy()
    dst = edges["cusip"].map(cusip_pos).to_numpy()
    edge_index = np.stack(
        [np.concatenate([src, dst]), np.concatenate([dst, src])], axis=0)
    w = edges["w"].to_numpy().astype(np.float32)
    aum_map = cik_df.set_index("cik")["log_aum"]
    aum = edges["cik"].map(aum_map).fillna(0.0).to_numpy().astype(np.float32)
    attr = (w * aum).astype(np.float32)
    if attr.size > 0 and float(attr.std()) > 0:
        attr = (attr - attr.mean()) / attr.std()
    edge_weight = np.concatenate([w, w])
    edge_attr = np.concatenate([attr, attr]).reshape(-1, 1)
    return edge_index, edge_weight, edge_attr


def _assign_labels(num_nodes, cusip_pos, cik_pos, r_next, prof_next):
    y = np.full(num_nodes, -1, dtype=np.int64)
    if not r_next.empty:
        stk_lbl = tertile_labels(r_next)
        for cusip, idx in cusip_pos.items():
            v = stk_lbl.get(cusip, -1)
            if v >= 0:
                y[idx] = int(v)
    if not prof_next.empty:
        inv_lbl = tertile_labels(prof_next)
        for cik, idx in cik_pos.items():
            v = inv_lbl.get(cik, -1)
            if v >= 0:
                y[idx] = int(v)
    return y


def build_graph(year, quarter, edges_col_name):
    edges = load_edges(year, quarter, edges_col_name)
    if edges.empty:
        raise RuntimeError(f"no Δ-edges for {year} Q{quarter}")
    cik_df, CIK_FEATS = _build_cik_features(edges, year, quarter)
    stock_df = _build_stock_features(year, quarter)
    r_next, prof_next = _load_label_sources(year, quarter)
    X, cik_ids, cusip_ids, cik_pos, cusip_pos = _assemble_node_matrix(
        edges, cik_df, stock_df, CIK_FEATS)
    edge_index, edge_weight, edge_attr = _assemble_edges(edges, cik_pos, cusip_pos, cik_df)
    y = _assign_labels(X.shape[0], cusip_pos, cik_pos, r_next, prof_next)

    data = Data(
        x=torch.tensor(X),
        edge_index=torch.tensor(edge_index, dtype=torch.long),
        edge_weight=torch.tensor(edge_weight),
        edge_attr=torch.tensor(edge_attr),
        y=torch.tensor(y),
    )
    data.is_cik = torch.zeros(X.shape[0], dtype=torch.bool)
    data.is_cik[:len(cik_ids)] = True
    data.has_label = data.y >= 0
    meta = {
        "cik_ids": cik_ids, "cusip_ids": cusip_ids,
        "n_cik": len(cik_ids), "n_cusip": len(cusip_ids),
    }
    return data, meta


### sanity check

In [ ]:
data, meta = build_graph(YEAR, QUARTER, EDGES_COLUMN_NAMES[0])
print(f"nodes: {data.num_nodes:,}  (CIKs {meta['n_cik']:,}, CUSIPs {meta['n_cusip']:,})")
print(f"edges: {data.num_edges:,}   edge_attr shape: {tuple(data.edge_attr.shape)}")
print(f"edge_weight range: [{data.edge_weight.min():.3f}, {data.edge_weight.max():.3f}]")
print(f"edge_attr   range: [{data.edge_attr.min():.3f}, {data.edge_attr.max():.3f}]")


## Models — `WeightedSAGE` (uses `edge_weight`) and `WeightedGAT` (`GATv2`, attention sees `w·log_aum`)

In [ ]:
class WeightedSAGE(nn.Module):
    """Mean-aggregating message-passing with edge_weight applied to messages.

    Uses SAGEConv when the installed PyG version supports edge_weight
    (>= 2.4); otherwise falls back to GraphConv, which has supported
    edge_weight since PyG 1.x and uses an equivalent aggregation.
    """
    def __init__(self, in_dim, hidden_dim, num_classes=NUM_CLASSES,
                 num_layers=2, dropout=0.5):
        super().__init__()
        Conv = SAGEConv if SAGE_SUPPORTS_EW else GraphConv
        self.convs = nn.ModuleList()
        if SAGE_SUPPORTS_EW:
            self.convs.append(Conv(in_dim, hidden_dim))
            for _ in range(num_layers - 1):
                self.convs.append(Conv(hidden_dim, hidden_dim))
        else:
            self.convs.append(Conv(in_dim, hidden_dim, aggr="mean"))
            for _ in range(num_layers - 1):
                self.convs.append(Conv(hidden_dim, hidden_dim, aggr="mean"))
        self.head = nn.Linear(hidden_dim, num_classes)
        self.dropout = dropout

    def forward(self, x, edge_index, edge_weight=None, edge_attr=None):
        for i, conv in enumerate(self.convs):
            if SAGE_SUPPORTS_EW:
                x = conv(x, edge_index, edge_weight=edge_weight)
            else:
                x = conv(x, edge_index, edge_weight)   # GraphConv: 3rd positional is edge_weight
            if i < len(self.convs) - 1:
                x = F.relu(x)
                x = F.dropout(x, p=self.dropout, training=self.training)
        return self.head(x)


class WeightedGAT(nn.Module):
    """GATv2 whose attention conditions on edge_attr = z-score(w * log_aum)."""
    def __init__(self, in_dim, hidden_dim, num_classes=NUM_CLASSES,
                 num_layers=2, heads=4, dropout=0.5, edge_dim=1):
        super().__init__()
        self.convs = nn.ModuleList()
        self.convs.append(GATv2Conv(in_dim, hidden_dim, heads=heads,
                                    dropout=dropout, edge_dim=edge_dim))
        for _ in range(num_layers - 2):
            self.convs.append(GATv2Conv(hidden_dim * heads, hidden_dim, heads=heads,
                                        dropout=dropout, edge_dim=edge_dim))
        self.convs.append(GATv2Conv(hidden_dim * heads, hidden_dim, heads=1,
                                    concat=False, dropout=dropout, edge_dim=edge_dim))
        self.head = nn.Linear(hidden_dim, num_classes)
        self.dropout = dropout

    def forward(self, x, edge_index, edge_weight=None, edge_attr=None):
        for i, conv in enumerate(self.convs):
            x = conv(x, edge_index, edge_attr=edge_attr)
            if i < len(self.convs) - 1:
                x = F.elu(x)
                x = F.dropout(x, p=self.dropout, training=self.training)
        return self.head(x)


## Training & evaluation (both `edge_weight` and `edge_attr` are now passed in)

In [ ]:
def train_one(model, data, train_mask, val_mask, epochs=EPOCHS, lr=LR, verbose=False):
    model = model.to(DEVICE)
    data = data.to(DEVICE)
    train_mask = train_mask.to(DEVICE)
    val_mask = val_mask.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)

    best_val_acc = 0.0
    best_state = None
    for ep in range(1, epochs + 1):
        model.train()
        opt.zero_grad()
        logits = model(data.x, data.edge_index, data.edge_weight, data.edge_attr)
        loss = F.cross_entropy(logits[train_mask], data.y[train_mask])
        loss.backward()
        opt.step()

        model.eval()
        with torch.no_grad():
            logits = model(data.x, data.edge_index, data.edge_weight, data.edge_attr)
            pred = logits.argmax(dim=-1)
            val_acc = (
                (pred[val_mask] == data.y[val_mask]).float().mean().item()
                if val_mask.any() else 0.0
            )
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        if verbose and ep % 25 == 0:
            print(f"  ep {ep:3d}  loss={loss.item():.4f}  val_acc={val_acc:.3f}")

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


def eval_subsets(model, data, mask):
    model.eval()
    with torch.no_grad():
        logits = model(
            data.x.to(DEVICE),
            data.edge_index.to(DEVICE),
            data.edge_weight.to(DEVICE),
            data.edge_attr.to(DEVICE),
        )
        pred = logits.argmax(dim=-1).cpu()
    y = data.y.cpu()
    out = {}
    for label, sel in [
        ("all", mask),
        ("stocks", mask & (~data.is_cik.cpu())),
        ("investors", mask & data.is_cik.cpu()),
    ]:
        if sel.sum() == 0:
            out[label] = {"n": 0}
            continue
        yt = y[sel].numpy()
        yp = pred[sel].numpy()
        out[label] = {
            "n": int(sel.sum()),
            "accuracy": accuracy_score(yt, yp),
            "macro_f1": f1_score(yt, yp, average="macro", labels=[0, 1, 2], zero_division=0),
        }
    return out


## CUSIP scoring helpers (P(top tertile) per CUSIP)

In [ ]:
def compute_cusip_scores(model, data, meta, year, quarter):
    """Score each test-graph CUSIP by P(top tertile next quarter). Returns
    a dataframe sorted descending: [cusip, year, quarter, score]."""
    model.eval()
    with torch.no_grad():
        logits = model(
            data.x.to(DEVICE),
            data.edge_index.to(DEVICE),
            data.edge_weight.to(DEVICE),
            data.edge_attr.to(DEVICE),
        )
        probs = F.softmax(logits, dim=-1).cpu().numpy()
    n_cik = meta["n_cik"]
    cusip_ids = meta["cusip_ids"]
    stock_probs = probs[n_cik:n_cik + len(cusip_ids)]
    df = pd.DataFrame({
        "cusip":   list(cusip_ids),
        "year":    int(year),
        "quarter": int(quarter),
        "score":   stock_probs[:, 2],
    })
    return df.sort_values("score", ascending=False).reset_index(drop=True)


def append_cusip_scores(df_new, parquet_path, year, quarter, model_tag):
    df_new = df_new.copy()
    df_new["model"] = model_tag
    if parquet_path.exists():
        prev = pd.read_parquet(parquet_path)
        prev = prev[~(
            (prev["year"] == year) &
            (prev["quarter"] == quarter) &
            (prev["model"] == model_tag)
        )]
        df_new = pd.concat([prev, df_new], ignore_index=True)
    df_new.to_parquet(parquet_path, index=False)


## Per-quarter sweep × two edge columns

For each `EDGES_COLUMN_NAME` in `[change_in_weight, change_in_adjusted_weight]`:
- run two-periods eval per quarter (resumable),
- write per-quarter rankings to `OUT_DIR/cusip_scores__<col>.parquet`,
- save best model as `models/best_<tag>_<year>Q<q>__<col>.pkl`,
- aggregate results CSV `models/two_periods_quarter_results__<col>.csv`.

In [ ]:
def two_periods_with_model(year, quarter, model_cls, edges_col_name, **kwargs):
    py, pq = prev_year_quarter(year, quarter)
    train_data, _ = build_graph(py, pq, edges_col_name)
    test_data, test_meta = build_graph(year, quarter, edges_col_name)

    Fdim = max(train_data.num_node_features, test_data.num_node_features)

    def pad(d):
        if d.num_node_features < Fdim:
            pad_cols = torch.zeros(d.num_nodes, Fdim - d.num_node_features)
            d.x = torch.cat([d.x, pad_cols], dim=1)
        return d

    train_data = pad(train_data)
    test_data = pad(test_data)

    model = model_cls(Fdim, HIDDEN_DIM, **kwargs)
    model = train_one(model, train_data, train_data.has_label, train_data.has_label, verbose=False)
    results = eval_subsets(model, test_data, test_data.has_label)
    return model, results, Fdim, test_data, test_meta


In [ ]:
import time
import pickle
import gc


def list_available_quarters(edges_col_name):
    df = CHANGED_HOLDINGS
    sub = df.loc[df[edges_col_name].notna(), ["year", "quarter"]].drop_duplicates()
    yq = sorted({(int(y), int(q)) for y, q in sub.itertuples(index=False)})
    avail = set(yq)
    out = []
    for y, q in yq:
        py, pq = prev_year_quarter(y, q)
        ny, nq = next_year_quarter(y, q)
        if (py, pq) in avail and (ny, nq) in avail:
            out.append((y, q))
    return out


def load_done_set(csv_path):
    if not csv_path.exists():
        return set()
    df = pd.read_csv(csv_path)
    return set(zip(df["year"].astype(int), df["quarter"].astype(int)))


def append_row(row, csv_path):
    pd.DataFrame([row]).to_csv(
        csv_path, mode="a", header=not csv_path.exists(), index=False)


def aggressive_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()


for EDGES_COLUMN_NAME in EDGES_COLUMN_NAMES:
    suffix = EDGES_COLUMN_NAME
    print(f"\n{'='*70}\n  SWEEP for edge column = {EDGES_COLUMN_NAME}\n{'='*70}")

    RESULTS_CSV = MODELS_DIR / f"two_periods_quarter_results__{suffix}.csv"
    SCORES_PARQUET = OUT_DIR / f"cusip_scores__{suffix}.parquet"

    quarters = list_available_quarters(EDGES_COLUMN_NAME)
    if not quarters:
        print(f"  no quarters found for {EDGES_COLUMN_NAME}, skipping")
        continue

    print(f"  {len(quarters)} quarters: {quarters[0]} … {quarters[-1]}")
    done = load_done_set(RESULTS_CSV)
    remaining = [(y, q) for y, q in quarters if (y, q) not in done]
    print(f"  resume: {len(done)} done, {len(remaining)} remaining")

    best = {"acc": -1.0, "model": None, "tag": None, "Fdim": None, "year": None, "quarter": None}
    t_start = time.perf_counter()

    for i, (y, q) in enumerate(remaining, 1):
        row = {"year": y, "quarter": q}

        for tag, model_cls, kw in [
            ("sage", WeightedSAGE, dict(num_layers=NUM_LAYERS, dropout=DROPOUT)),
            ("gat",  WeightedGAT,  dict(num_layers=NUM_LAYERS, heads=GAT_HEADS, dropout=DROPOUT)),
        ]:
            try:
                if torch.cuda.is_available():
                    torch.cuda.reset_peak_memory_stats()
                t0 = time.perf_counter()
                m, r, fd, test_data, test_meta = two_periods_with_model(
                    y, q, model_cls, EDGES_COLUMN_NAME, **kw)
                row[f"{tag}_train_s"] = time.perf_counter() - t0
                row[f"{tag}_peak_gb"] = (torch.cuda.max_memory_allocated() / 1e9
                                           if torch.cuda.is_available() else 0.0)
                row[f"{tag}_acc"] = r["all"].get("accuracy")
                row[f"{tag}_f1"] = r["all"].get("macro_f1")
                row[f"{tag}_stocks_acc"] = r["stocks"].get("accuracy")
                row[f"{tag}_inv_acc"] = r["investors"].get("accuracy")

                # per-quarter CUSIP score parquet
                scores_df = compute_cusip_scores(m, test_data, test_meta, y, q)
                append_cusip_scores(scores_df, SCORES_PARQUET, y, q, tag)

                if row[f"{tag}_acc"] is not None and row[f"{tag}_acc"] > best["acc"]:
                    best.update({"acc": row[f"{tag}_acc"], "model": m.cpu(), "tag": tag,
                                  "Fdim": fd, "year": y, "quarter": q})
            except Exception as e:
                print(f"  ! {tag.upper()} {y} Q{q} failed: {e.__class__.__name__}: {e}")
            finally:
                aggressive_cleanup()

        append_row(row, RESULTS_CSV)
        elapsed = time.perf_counter() - t_start
        eta = elapsed / max(i, 1) * (len(remaining) - i)
        free_gb = (torch.cuda.mem_get_info()[0] / 1e9 if torch.cuda.is_available() else 0.0)
        print(f"  [{len(done) + i:2d}/{len(quarters)}] {y} Q{q}  "
              f"SAGE={row.get('sage_acc', float('nan')):.3f}  "
              f"GAT={row.get('gat_acc', float('nan')):.3f}  "
              f"free {free_gb:.2f}GB  ETA {eta/60:.1f}m")

    if best["model"] is not None:
        best_path = MODELS_DIR / f"best_{best['tag']}_{best['year']}Q{best['quarter']}__{suffix}.pkl"
        with open(best_path, "wb") as f:
            pickle.dump({
                "tag": best["tag"], "year": best["year"], "quarter": best["quarter"],
                "Fdim": best["Fdim"], "test_acc": best["acc"],
                "state_dict": {k: v.cpu() for k, v in best["model"].state_dict().items()},
                "config": {
                    "HIDDEN_DIM": HIDDEN_DIM, "NUM_LAYERS": NUM_LAYERS,
                    "GAT_HEADS": GAT_HEADS, "DROPOUT": DROPOUT,
                    "NUM_CLASSES": NUM_CLASSES, "EDGES_COLUMN_NAME": EDGES_COLUMN_NAME,
                },
            }, f)
        print(f"  saved best -> {best_path.name}")

    print(f"  finished {EDGES_COLUMN_NAME} in {(time.perf_counter() - t_start)/60:.1f} min")
    print(f"  scores parquet -> {SCORES_PARQUET.name}")


## Aggregate summary across both runs

In [ ]:
for EDGES_COLUMN_NAME in EDGES_COLUMN_NAMES:
    suffix = EDGES_COLUMN_NAME
    csv_path = MODELS_DIR / f"two_periods_quarter_results__{suffix}.csv"
    if not csv_path.exists():
        print(f"[skip] {csv_path.name} not found"); continue
    df = pd.read_csv(csv_path)
    print(f"\n=== {EDGES_COLUMN_NAME}  ({len(df)} quarters) ===")
    print(df[["sage_acc", "gat_acc"]].agg(["mean", "std"]).T.round(3))
    print(df[["sage_f1", "gat_f1"]].agg(["mean", "std"]).T.round(3))
